# 🌱 Lesson 7: Scikit-learn Pipelines: Everything Together
**Phase 2 · Data Preprocessing & Feature Engineering** &nbsp;|&nbsp; ⏱ about 3–4 hours &nbsp;|&nbsp; 📚 Prerequisite: Lessons 01–06

> *A pipeline is a recipe that cooks the same meal every time: for the test set and for tomorrow's data.*

## 🎯 By the end of this lesson you will be able to
- Chain imputation, scaling, encoding, feature creation, selection and a model into one object.
- Route different columns to different treatments with `ColumnTransformer`.
- Write custom transformers that obey the `fit` / `transform` contract.
- Tune preprocessing choices with `GridSearchCV` without leakage.
- Inspect, save, reload and reuse a pipeline on messy new data.

---

In [1]:
import sys
sys.path.append("../utils")            # lets us import course_utils.py and agri_prep.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils import *             # load_raw_farm, profile, setup_plots, ...

setup_plots()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(42)

# Part 1 · Why pipelines?

So far we've done many steps *by hand*: impute, cap, scale, encode, build features, select. Every manual step is a place for the three classic accidents:

| Accident | Cause | What a pipeline does |
|---|---|---|
| **Leakage** | You fit a step on all data | Re-fits every step on training data only, automatically |
| **Inconsistency** | Test data processed slightly differently from training data | Applies *exactly* the same learned transformations |
| **Irreproducibility** | "Which cells did I run, in what order?" | One object = the entire recipe |

> **Analogy: the assembly line.** Each station (imputer, scaler, encoder, model) does one job and hands the product to the next. You don't carry parts around by hand. You switch the line on.

### Our dataset for this lesson
The structurally-clean farm records (Lessons 1–2), **without** the columns we ruled out in Lesson 6 (`harvest_date`, price and revenue; none is known at planting time) and the identifier `farm_id`. The target is `log(yield)`; we'll evaluate with `GroupKFold` by farm.

In [2]:
from sklearn.model_selection import GroupKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge
from agri_prep import DateFeatures, AgriFeatures, Winsorizer

df = load_lesson_clean()
LEAKY = ["total_revenue_ksh", "market_price_ksh_kg"]          # not known at planting time (Lesson 6)
X = df.drop(columns=["yield_t_ha", "farm_id", "season", "harvest_date"] + LEAKY)   # harvest_date: not known at planting
y = np.log(df["yield_t_ha"])                                  # we model log(yield)
groups = df["farm_id"]

NUM = ["rainfall_mm", "avg_temp_c", "soil_ph", "nitrogen_ppm", "phosphorus_ppm", "potassium_ppm", "organic_matter_pct",
       "fertilizer_kg_ha", "irrigation", "plant_year", "plant_month_sin", "plant_month_cos", "log_farm_size",
       "npk_total", "n_to_p_ratio", "moisture_index", "fert_x_irrigation", "rain_gap_sq", "temp_gap_sq", "ph_gap_sq"]
NOM = ["crop", "county", "soil_type", "rain_season"]
ORD = ["experience", "pest_pressure"]
ORD_CATS = [["Beginner", "Intermediate", "Advanced", "Expert"], ["Low", "Medium", "High"]]


def build_preprocessor(strategy="median", cap_k=3.0):
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy=strategy)), ("cap", Winsorizer(k=cap_k)), ("scale", StandardScaler())]), NUM),
        ("nom", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), NOM),
        ("ord", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("ordinal", OrdinalEncoder(categories=ORD_CATS, handle_unknown="use_encoded_value", unknown_value=-1))]), ORD),
    ], remainder="drop").set_output(transform="pandas")


def build_pipeline(model=None, k="all", strategy="median", cap_k=3.0):
    return Pipeline([("dates", DateFeatures(harvest_col=None)), ("feats", AgriFeatures()), ("prep", build_preprocessor(strategy, cap_k)),
                     ("select", SelectKBest(f_regression, k=k)), ("model", model if model is not None else Ridge(alpha=1.0))])

In [3]:
print("X:", X.shape, "| target: log(yield) |", groups.nunique(), "farms")
X.head(3)

X: (2329, 16) | target: log(yield) | 600 farms


,county,soil_type,farm_size_ha,experience,crop,planting_date,rainfall_mm,avg_temp_c,soil_ph,nitrogen_ppm,phosphorus_ppm,potassium_ppm,organic_matter_pct,fertilizer_kg_ha,irrigation,pest_pressure
0,Machakos,Clay,1.06,Advanced,Potato,2023-04-06,491.1,21.7,6.61,27.6,11.0,226.4,1.90,66.0,1,High
1,Kakamega,Clay,0.77,Advanced,Sorghum,2021-11-09,645.8,19.6,5.83,17.4,18.5,134.6,2.68,NaN,1,Medium
2,Kisii,Volcanic,1.69,Intermediate,Potato,2019-03-16,905.6,16.6,5.58,18.6,13.4,101.3,2.55,69.0,1,Low


---
# Part 2 · Pipeline basics

A `Pipeline` is a list of `(name, step)` pairs. All steps except the last must be *transformers* (have `fit` and `transform`); the last can be a *model* (`fit` and `predict`).

```
raw data ──► [imputer] ──► [scaler] ──► [model] ──► prediction
              fit_transform  fit_transform  fit            (during .fit)
              transform      transform      predict        (during .predict)
```

### The smallest possible pipeline

In [4]:
cols = ["rainfall_mm", "avg_temp_c", "soil_ph", "nitrogen_ppm", "organic_matter_pct", "fertilizer_kg_ha"]
pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge())
pipe          # in Jupyter this renders as a diagram: click it!

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('simpleimputer', ...), ('standardscaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive est

In [5]:
tr, te = next(GroupKFold(5).split(X, groups=groups))
pipe.fit(X.iloc[tr][cols], y.iloc[tr])                  # ONE call: fits imputer → scaler → model on TRAIN
print("test R²:", round(pipe.score(X.iloc[te][cols], y.iloc[te]), 3))     # ONE call: imputes → scales → predicts → scores

print("\nsteps          :", list(pipe.named_steps))
print("learned medians:", pipe["simpleimputer"].statistics_.round(2))      # what the imputer learned from TRAIN
print("learned means  :", pipe["standardscaler"].mean_.round(2))

test R²: 0.079

steps          : ['simpleimputer', 'standardscaler', 'ridge']
learned medians: [671.9   18.8    6.02  29.7    3.21  70.  ]
learned means  : [665.32  18.96   6.02  30.2    3.2   68.46]


Notice what *wasn't* needed: no manual `fit_transform` on train, `transform` on test. The pipeline follows the golden rule for us.

### Peeking inside: slicing
`pipe[:-1]` is "everything except the model": great for looking at what the model actually receives:

In [6]:
pipe[:-1].transform(X.iloc[te][cols]).round(2)[:3]

array([[-0.97,  1.28,  1.04, -0.17, -1.39, -0.06],
       [ 1.98, -0.31,  0.52, -0.54, -1.88, -0.17],
       [-0.82, -0.87,  0.4 , -0.83,  0.77,  1.18]])

### Cross-validation: the payoff
Pass the **whole pipeline** to `cross_val_score`. In every fold the imputer, the scaler *and* the model are re-fitted on that fold's training rows only. No leakage is possible by construction.

In [7]:
scores = cross_val_score(pipe, X[cols], y, cv=GroupKFold(5), groups=groups, scoring="r2")
print("CV R² per fold:", scores.round(3), "| mean:", scores.mean().round(3))

CV R² per fold: [0.079 0.098 0.075 0.075 0.077] | mean: 0.081


The score is modest: we gave the model six raw columns and **no crop information**: crop is the biggest driver of yield. Let's give each kind of column the right treatment.

---
# Part 3 · `ColumnTransformer`: different columns, different recipes

| Column group | Recipe |
|---|---|
| Numeric | impute (median) → scale |
| Nominal categories | impute (mode) → one-hot |
| Ordinal categories | impute (mode) → ordinal with explicit order |

In [8]:
ct = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["rainfall_mm", "avg_temp_c", "soil_ph"]),
    ("nom", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", sparse_output=False)), ["crop", "county"]),
    ("ord", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OrdinalEncoder(categories=[["Low", "Medium", "High"]], handle_unknown="use_encoded_value", unknown_value=-1)), ["pest_pressure"]),
]).set_output(transform="pandas")             # ← output a DataFrame with readable column names

Xt = ct.fit_transform(X.iloc[tr])
print(X.iloc[tr].shape, "→", Xt.shape, "| NaN left:", int(Xt.isna().sum().sum()))
Xt.iloc[:3, :8].round(2)

(1863, 16) → (1863, 19) | NaN left: 0


,num__rainfall_mm,num__avg_temp_c,num__soil_ph,nom__crop_Beans,nom__crop_Maize,nom__crop_Potato,nom__crop_Sorghum,nom__crop_Wheat
1,-0.11,0.30,-0.34,0.0,0.0,0.0,1.0,0.0
2,1.34,-1.10,-0.79,0.0,0.0,1.0,0.0,0.0
3,1.56,0.49,-1.64,0.0,1.0,0.0,0.0,0.0


Column names carry a **prefix** telling you which branch produced them (`num__`, `nom__`, `ord__`). Four mixed-type columns became 17 clean, numeric, NaN-free columns. `remainder='drop'` (the default) discards any column you didn't mention; `remainder='passthrough'` keeps them untouched. Prefer *explicit lists*: then nothing sneaks into the model by accident.

---
# Part 4 · Writing your own transformer

Sometimes no built-in does what you need. A transformer is any class with:

```python
class MyTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, param=1):        # ONLY store parameters. No computation.
        self.param = param
    def fit(self, X, y=None):           # LEARN from training data; save as attributes ending in "_"
        self.something_ = ...
        return self                     # always return self
    def transform(self, X):             # APPLY what was learned. Return a NEW object.
        ...
```

### Example: a Winsorizer (cap outliers) that learns its fences from training data
In Lesson 2 we decided that statistical capping must be learned on training data only. Here is the transformer that makes that automatic:

In [9]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class SimpleWinsorizer(BaseEstimator, TransformerMixin):
    """Clip each column to  [Q1 - k*IQR,  Q3 + k*IQR]  learned from the data passed to fit()."""
    def __init__(self, k=1.5):
        self.k = k                                           # parameter only

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        q1, q3 = X.quantile(0.25), X.quantile(0.75)
        iqr = q3 - q1
        # guard: a zero IQR (e.g. a 0/1 flag) would collapse the fences onto one value and erase the column
        self.lower_ = (q1 - self.k * iqr).where(iqr > 0, -np.inf)        # learned state: trailing underscore
        self.upper_ = (q3 + self.k * iqr).where(iqr > 0, np.inf)
        return self

    def transform(self, X):
        check_is_fitted(self)                                # helpful error if fit() was forgotten
        return pd.DataFrame(X).clip(self.lower_, self.upper_, axis=1)

# Try it: fences come from TRAIN, but test values are clipped by the SAME fences
col = ["farm_size_ha"]
train_col, test_col = X.iloc[tr][col], X.iloc[te][col]
w = SimpleWinsorizer(k=3).fit(train_col)
print(f"fences learned on train: [{w.lower_.iloc[0]:.2f}, {w.upper_.iloc[0]:.2f}] ha")
print(f"test max before: {test_col.max().iloc[0]:.1f} ha → after: {w.transform(test_col).max().iloc[0]:.1f} ha")

fences learned on train: [-3.30, 5.87] ha
test max before: 338.8 ha → after: 5.9 ha


> 🐞 **A real bug this guard prevents.** Our first version had no `iqr > 0` check. In the farm data about 75% of farms are *not* irrigated, so for the 0/1 `irrigation` column Q1 = Q3 = 0, the IQR is 0, and the "fences" were [0, 0]: the Winsorizer silently flipped every 1 to 0 and the column became constant! The project's `check_ml_ready` self-test ("no constant columns") caught it. Moral: **test every custom transformer on binary and near-constant columns**, and always run sanity checks on the matrix that reaches the model.

Compare with `StandardScaler` or any built-in: same contract. That means **any** custom step slots into `Pipeline`, `cross_val_score` and `GridSearchCV` like a native one.

For this course we've placed three production-grade transformers in **`utils/agri_prep.py`**:

| Class | Job |
|---|---|
| `DateFeatures` | planting date → year, month, sin/cos month, day-of-year, `rain_season` (and `grow_days` if a harvest date is supplied) |
| `AgriFeatures` | ratios, sums, log size, fertiliser × irrigation, distance-from-optimum features |
| `Winsorizer` | IQR capping learned on training data (like the one above, with `get_feature_names_out`) |

> **Why a module?** Saved pipelines (joblib/pickle) store the *name* of each class. A class defined in a notebook lives in `__main__`; reload the file in another script and Python can't find it. Put reusable transformers in a `.py` file.

In [10]:
tmp = AgriFeatures().fit_transform(DateFeatures(harvest_col=None).fit_transform(X.head(4)))
tmp[["crop", "plant_year", "plant_month_sin", "plant_dayofyear", "rain_season", "npk_total", "moisture_index", "rain_gap_sq"]].round(2)

,crop,plant_year,plant_month_sin,plant_dayofyear,rain_season,npk_total,moisture_index,rain_gap_sq
0,Potato,2023,0.87,96,Long,265.0,22.63,11859.21
1,Sorghum,2021,-0.50,313,Short,170.5,32.95,60417.64
2,Potato,2019,1.00,75,Long,133.3,54.55,93391.36
3,Maize,2019,1.00,66,Long,187.7,47.22,86671.36


---
# Part 5 · The complete pipeline

Now we assemble *everything*:

```
raw rows ─► DateFeatures ─► AgriFeatures ─► ColumnTransformer ───────────────► SelectKBest ─► Ridge
            (dates→numbers)  (new features)  num: impute → cap → scale          (inside CV)
                                             nom: impute → one-hot
                                             ord: impute → ordinal
```
`build_pipeline()` (defined in the setup cell) wires this together. Render it:

In [11]:
from sklearn import set_config
set_config(display="diagram")
pipe = build_pipeline()
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dates', ...), ('feats', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,harvest_col,None
,plant_col,'planting_date'
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_thre

### How well does it do? (and how much did all this work matter?)

In [12]:
cv = GroupKFold(5)
crude = make_pipeline(SimpleImputer(strategy="mean"), Ridge())          # raw numeric columns only; no crop, no engineering
raw_num = ["rainfall_mm", "avg_temp_c", "soil_ph", "nitrogen_ppm", "phosphorus_ppm", "potassium_ppm", "organic_matter_pct", "fertilizer_kg_ha", "farm_size_ha"]

from sklearn.ensemble import RandomForestRegressor
results = {
    "crude: raw numeric columns → Ridge":        cross_val_score(crude, X[raw_num], y, cv=cv, groups=groups, scoring="r2").mean(),
    "full pipeline → Ridge":                     cross_val_score(build_pipeline(), X, y, cv=cv, groups=groups, scoring="r2").mean(),
    "full pipeline → Random Forest":             cross_val_score(build_pipeline(RandomForestRegressor(150, min_samples_leaf=3, n_jobs=-1, random_state=0)), X, y, cv=cv, groups=groups, scoring="r2").mean(),
}
pd.Series(results).round(3).to_frame("CV R² (log yield)")

,CV R² (log yield)
crude: raw numeric columns → Ridge,0.079
full pipeline → Ridge,0.932
full pipeline → Random Forest,0.933


There is the payoff of Phase 2: the same simple linear model goes from a poor fit to an excellent one purely through preparation: cleaning, encoding, engineering, scaling. And the pipeline is **leak-proof** because it is the thing being cross-validated.

### Looking inside: what did it learn?

In [13]:
pipe.fit(X, y)
names = pipe["prep"].get_feature_names_out()                      # names after the ColumnTransformer
print("features entering the model:", len(names))

coef = pd.Series(pipe["model"].coef_, index=names[pipe["select"].get_support()])
coef.reindex(coef.abs().sort_values().index).tail(8).round(3).to_frame("Ridge coefficient (largest by size)")

features entering the model: 44


,Ridge coefficient (largest by size)
ord__pest_pressure,-0.174
num__ph_gap_sq,-0.339
num__temp_gap_sq,-0.369
nom__crop_Maize,0.439
num__rain_gap_sq,-0.642
nom__crop_Sorghum,-1.046
nom__crop_Beans,-1.160
nom__crop_Potato,1.637


Because numeric inputs are standardised, coefficients are comparable: the distance-from-optimum features and the crop indicators dominate: the same story Lessons 4 and 5 told, now confirmed by a model.

---
# Part 6 · Tuning preprocessing with `GridSearchCV`

Since preprocessing *is* part of the pipeline, its choices become hyper-parameters. Parameter names are `step__substep__parameter`:

In [14]:
[k for k in build_pipeline().get_params() if k.endswith(("strategy", "__k", "alpha"))]

['prep__num__impute__strategy',
 'prep__num__cap__k',
 'prep__nom__impute__strategy',
 'prep__ord__impute__strategy',
 'select__k',
 'model__alpha']

In [15]:
grid = {"prep__num__impute__strategy": ["mean", "median"],
        "prep__num__cap__k": [1.5, 10.0],
        "select__k": [20, "all"],
        "model__alpha": [0.1, 1, 10]}

gs = GridSearchCV(build_pipeline(), grid, cv=GroupKFold(4), scoring="r2", n_jobs=-1).fit(X, y, groups=groups)
print("best params:", gs.best_params_)
print("best CV R² :", round(gs.best_score_, 4))

res = pd.DataFrame(gs.cv_results_).sort_values("rank_test_score")
res[[c for c in res.columns if c.startswith("param_")] + ["mean_test_score"]].head(5).round(4).reset_index(drop=True)

best params: {'model__alpha': 1, 'prep__num__cap__k': 10.0, 'prep__num__impute__strategy': 'mean', 'select__k': 'all'}
best CV R² : 0.9402


,param_model__alpha,param_prep__num__cap__k,param_prep__num__impute__strategy,param_select__k,mean_test_score
0,1.0,10.0,mean,all,0.9402
1,0.1,10.0,mean,all,0.9401
2,10.0,10.0,mean,all,0.9398
3,1.0,10.0,median,all,0.9393
4,0.1,10.0,median,all,0.9393


All combinations are scored with the *entire* preprocessing re-fitted inside each fold: a feature-selection `k` is tuned **without** the selection leakage we exposed in Lesson 6.

> 🧭 Two things to notice. **(1)** Mean vs median imputation and the Ridge strength barely matter. **(2)** The capping multiplier *does*: aggressive capping (`k=1.5`) scores clearly worse than gentle capping (`k=10`). Why? Engineered features like `rain_gap_sq` are naturally long-tailed, and their large values are *real signal* (a very dry season really does hurt yield). Outlier treatment is not free: it must be **tested, not assumed**. And don't over-tune: more search = more chance to fit noise.

---
# Part 7 · A final, honest test

So far we cross-validated. Final protocol: hold out a **test set once**, untouched, split **by farm**, fit the pipeline on the rest, report a single number.

In [16]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score

tr_i, te_i = next(GroupShuffleSplit(test_size=0.2, random_state=42).split(X, groups=groups))
final = build_pipeline().set_params(**gs.best_params_)                                   # best settings from the search
final.fit(X.iloc[tr_i], y.iloc[tr_i])

pred = np.exp(final.predict(X.iloc[te_i]))                               # back to tonnes per hectare
true = np.exp(y.iloc[te_i])
print(f"held-out farms: {len(set(groups.iloc[te_i]))} | R² (t/ha): {r2_score(true, pred):.3f} | MAE: {mean_absolute_error(true, pred):.2f} t/ha")

held-out farms: 120 | R² (t/ha): 0.868 | MAE: 0.55 t/ha


Why is the held-out R² (≈ 0.87, in tonnes per hectare) lower than the cross-validated R² (≈ 0.94, on the log scale)? They are *different yardsticks*: on the tonnes scale, mistakes on high-yielding potatoes weigh far more than mistakes on beans. Always say which scale a score refers to, and report an error in units people understand (here the **MAE in t/ha**).

---
# Part 8 · Save it, reload it, use it on messy new data

A fitted pipeline *is* the deployable model: it contains every median, category list, fence and coefficient.

In [17]:
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)
joblib.dump(final, "../models/lesson07_pipeline.joblib")
loaded = joblib.load("../models/lesson07_pipeline.joblib")

new = X.iloc[[0]].copy()
new["county"] = "Narok"            # a county never seen in training
new["soil_ph"] = np.nan            # a missing number
new["pest_pressure"] = np.nan      # a missing category
print("prediction for a messy new record: %.2f t/ha" % np.exp(loaded.predict(new))[0])
print("identical to the original on 5 rows:", np.allclose(final.predict(X.head(5)), loaded.predict(X.head(5))))

prediction for a messy new record: 4.30 t/ha
identical to the original on 5 rows: True


A raw record with an unseen county and two missing values flows through: the missing number takes the training median, the missing label the training mode, the unseen county becomes all-zeros in the one-hot block. **This is what "production-ready preprocessing" means.**

### 🧾 Documenting your pipeline
A pipeline is code, but code is not documentation. For every step keep a record of *what, why, and learned-from-what*. A skeleton to copy into your project:

| Step | What it does | Why | Learned from training data? |
|---|---|---|---|
| `DateFeatures` | planting date → year, sin/cos month, day-of-year, rain_season | Season drives yield | No (rules) |
| `AgriFeatures` | ratios, log size, gap-to-optimum | Capture hill-shaped effects | No (rules) |
| Imputer (median) | fills missing numerics | Models can't take NaN | **Yes**: medians |
| Winsorizer (k=3) | caps extremes at Q1−3·IQR / Q3+3·IQR | Limit leverage of outliers | **Yes**: fences |
| StandardScaler | mean 0 / std 1 | Linear models need comparable scales | **Yes**: mean, std |
| OneHotEncoder | crop, county, soil, season → 0/1 columns | No false ordering | **Yes**: category list |
| OrdinalEncoder | experience, pest pressure → 0..k | Real order | No (explicit order) |
| SelectKBest | keep the k strongest features | Reduce noise | **Yes**: scores |

That last column is the heart of the leakage rule: every "Yes" must be learned on training data alone.

Your final project asks you to produce exactly this kind of documentation for the agricultural dataset.

---
## 🧩 Practice problems
Now it is your turn. Each problem is **self-contained** (it tells you what data to load), so you can do them in any order.

**How to work like a pro:**
1. Read the problem twice. Predict the answer *before* you code.
2. Struggle for at least 10 minutes. Struggling is where learning happens.
3. Open a **hint** only when stuck - they get progressively stronger.
4. Compare with `solutions/solutions_07.ipynb` **after** you have your own answer.

Difficulty: ⭐ warm-up · ⭐⭐ core skill · ⭐⭐⭐ challenge

### Problem 7.1 &nbsp; ⭐ &nbsp; Your first pipeline

Using only these raw numeric columns of `X`: `rainfall_mm, avg_temp_c, soil_ph, nitrogen_ppm, organic_matter_pct, fertilizer_kg_ha` (they contain NaN), build a pipeline **median imputer → StandardScaler → Ridge** and cross-validate it (5-fold `GroupKFold` on farms) against `y`.

Print the CV R² and the names of the pipeline steps. Why is it safe to pass raw data with NaN straight into `cross_val_score`?

<details><summary>💡 Hint 1</summary>

`make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge())`.

</details>

<details><summary>💡💡 Hint 2</summary>

`cross_val_score(pipe, X[cols], y, cv=GroupKFold(5), groups=groups, scoring='r2')`.

</details>

<details><summary>💡💡💡 Hint 3</summary>

`list(pipe.named_steps)` shows the automatically generated step names.

</details>

In [18]:
# ✏️ Your code here

### Problem 7.2 &nbsp; ⭐⭐ &nbsp; Mixed columns with ColumnTransformer

Build a `ColumnTransformer` that (a) median-imputes and scales `rainfall_mm`, `avg_temp_c`, `soil_ph`; (b) one-hot encodes `crop` and `county` (`handle_unknown='ignore'`); (c) encodes `pest_pressure` ordinally (Low < Medium < High) after a most-frequent imputation.

Fit on the training half of a `GroupKFold` split, transform both halves, and report the output shape, the feature names, and the number of NaN left.

<details><summary>💡 Hint 1</summary>

Use three tuples `(name, transformer, columns)`. Wrap imputer+encoder in `make_pipeline` for (b) and (c).

</details>

<details><summary>💡💡 Hint 2</summary>

`ColumnTransformer(...).set_output(transform='pandas')` gives a DataFrame with readable names.

</details>

<details><summary>💡💡💡 Hint 3</summary>

`tr, te = next(GroupKFold(2).split(X, groups=groups))` gives index arrays; use `X.iloc[tr]`.

</details>

In [19]:
# ✏️ Your code here

### Problem 7.3 &nbsp; ⭐⭐ &nbsp; Write your own transformer

Write a scikit-learn transformer `LogColumns(columns)` that applies `np.log1p` to the given columns of a DataFrame and leaves the others untouched.

1. It must inherit `BaseEstimator` and `TransformerMixin`, store `columns` in `__init__`, implement `fit` (returning `self`) and `transform` (returning a **copy**).
2. Apply it to `farm_size_ha` and report the skewness before/after.
3. Put it into a `Pipeline` in front of a `SimpleImputer` and a `Ridge` using `['farm_size_ha','rainfall_mm']` and check that it cross-validates.

<details><summary>💡 Hint 1</summary>

`class LogColumns(BaseEstimator, TransformerMixin): def __init__(self, columns): self.columns = columns`.

</details>

<details><summary>💡💡 Hint 2</summary>

In `transform`: `X = X.copy(); X[self.columns] = np.log1p(X[self.columns]); return X`.

</details>

<details><summary>💡💡💡 Hint 3</summary>

`fit` has nothing to learn, but it must still exist and `return self`.

</details>

In [20]:
# ✏️ Your code here

### Problem 7.4 &nbsp; ⭐⭐ &nbsp; Tune preprocessing choices with GridSearchCV

Using the `build_pipeline()` helper from the lesson setup, run a `GridSearchCV` (`GroupKFold(4)`, `scoring='r2'`) over:

- the imputation **strategy** of the numeric branch: `mean`, `median`;
- the **cap multiplier** of the Winsorizer: `1.5`, `3.0`, `10.0`;
- the Ridge **alpha**: `0.1`, `1`, `10`.

Report the best parameters and score. Which preprocessing choice mattered most (compare the best and worst combination)?

<details><summary>💡 Hint 1</summary>

Parameter names follow the pattern `step__substep__param`: e.g. `prep__num__impute__strategy`, `prep__num__cap__k`, `model__alpha`.

</details>

<details><summary>💡💡 Hint 2</summary>

`GridSearchCV(pipe, grid, cv=GroupKFold(4), scoring='r2', n_jobs=-1).fit(X, y, groups=groups)`.

</details>

<details><summary>💡💡💡 Hint 3</summary>

`pd.DataFrame(gs.cv_results_)`: sort by `rank_test_score` and look at the extremes.

</details>

In [21]:
# ✏️ Your code here

### Problem 7.5 &nbsp; ⭐⭐⭐ &nbsp; Ship it: save, reload, and predict on messy new data

1. Fit `build_pipeline()` on the full training data.
2. Save it with `joblib.dump` to `../models/problem7_5.joblib`, then reload it.
3. Create **one new raw record** copied from the data, but with `county='Narok'` (never seen), `soil_ph` set to `NaN`, and `pest_pressure` set to `NaN`. Predict its yield (in t/ha) with the reloaded pipeline.
4. Confirm the reloaded pipeline gives identical predictions to the original on the first 5 rows, and explain why the prediction worked despite the problems in the new record.

<details><summary>💡 Hint 1</summary>

`Path('../models').mkdir(exist_ok=True)`; `joblib.dump(pipe, path)`; `joblib.load(path)`.

</details>

<details><summary>💡💡 Hint 2</summary>

`new = X.iloc[[0]].copy(); new['county'] = 'Narok'; new['soil_ph'] = np.nan; new['pest_pressure'] = np.nan`.

</details>

<details><summary>💡💡💡 Hint 3</summary>

Remember the model predicts `log(yield)`, so convert back with `np.exp`.

</details>

In [22]:
# ✏️ Your code here

---
## ✅ Check your understanding
Answer in your head first, *then* click to reveal.

**Q1.** What is the main safety benefit of passing a Pipeline to `cross_val_score`?

- **A)** It makes the code shorter
- **B)** Preprocessing is re-fitted on each training fold only, so validation rows never influence it
- **C)** It runs faster on GPUs
- **D)** It removes outliers

<details><summary>Show answer</summary>

**B**: Fitting inside each fold is exactly what prevents leakage during cross-validation.

</details>

**Q2.** In a custom transformer, where should values learned from the data be stored?

- **A)** In `__init__`
- **B)** In attributes ending with an underscore, set inside `fit`
- **C)** In global variables
- **D)** Inside `transform` each time

<details><summary>Show answer</summary>

**B**: Convention: constructor parameters in `__init__`, learned state in `fit` as `name_`. `transform` only *applies* it.

</details>

**Q3.** `handle_unknown='ignore'` in OneHotEncoder is important because…

- **A)** It sorts categories
- **B)** A category unseen in training would otherwise crash prediction
- **C)** It removes missing values
- **D)** It scales the output

<details><summary>Show answer</summary>

**B**: In production new categories appear; the encoder should degrade gracefully.

</details>

**Q4.** Why keep custom transformer classes in a `.py` module rather than only in the notebook?

- **A)** Notebooks are slow
- **B)** A saved pipeline can only be reloaded if Python can import its classes
- **C)** Modules are faster
- **D)** To hide the code

<details><summary>Show answer</summary>

**B**: Pickle stores a reference to the class by module name. Classes defined in a notebook live in `__main__` and break when reloaded elsewhere.

</details>

---
## 📌 Summary: what to remember
- A `Pipeline` chains transformers and a model; `.fit` / `.predict` run the whole assembly line.
- Put *everything learned from data* inside the pipeline; pass the pipeline to `cross_val_score`/`GridSearchCV`.
- `ColumnTransformer` gives each column group its own recipe; `set_output(transform='pandas')` keeps names readable.
- Custom transformer = `fit` (learn, store `name_`) + `transform` (apply, return a copy) + parameters in `__init__`.
- Keep custom classes in a module so saved pipelines can be reloaded.
- `handle_unknown='ignore'` + imputers make the pipeline robust to messy production data.
- Document each step: what, why, and whether it learns from training data.

## 🚧 Common mistakes
- ⚠️ Doing preprocessing outside the pipeline 'just to explore' and forgetting to redo it inside.
- ⚠️ Computation in `__init__` of a custom transformer (breaks `clone`/grid search).
- ⚠️ Modifying the input DataFrame in place inside `transform`.
- ⚠️ IQR-based capping on binary or mostly-zero columns (IQR = 0 collapses the fences; guard against it).
- ⚠️ Selecting columns by *position* instead of name in a ColumnTransformer.
- ⚠️ Tuning on the final test set.

➡️ **Next:** the **capstone project**: prepare the agricultural dataset for machine learning, end to end (`project/`).